In [ ]:
import requests

slug = "presidential-election-winner-2024"
url = f"https://gamma-api.polymarket.com/events?slug={slug}"
data = requests.get(url).json()

# Print all markets (one per candidate)
for market in data[0]['markets']:
    print(market['conditionId'], market['question'])

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

# ─── STEP 1: Fetch token_id via Gamma API (NOT conditionId!) ───
slug = "presidential-election-winner-2024"
gamma_url = f"https://gamma-api.polymarket.com/events?slug={slug}"
event = requests.get(gamma_url).json()[0]

# Print all candidates and their token IDs
for market in event['markets']:
    tokens = market.get('clobTokenIds', '[]')
    print(f"{market['question'][:50]} → tokens: {tokens}")

In [ ]:
event

In [ ]:
# ─── STEP 2: Select the correct token (Trump = Yes token) ───
import json

trump_market = next(
    m for m in event['markets'] 
    if 'Trump' in m['question']
)

# clobTokenIds contains [yes_token_id, no_token_id]
token_ids = json.loads(trump_market['clobTokenIds'])
trump_yes_token = token_ids[0]
print(f"Trump YES token_id: {trump_yes_token}")

In [ ]:
# ─── STEP 3: Fetch price history (correct approach for closed markets) ───
url = "https://clob.polymarket.com/prices-history"

params = {
    "market": trump_yes_token,   # ← token_id, NOT conditionId
    "interval": "max",           # ← required for closed markets
    "fidelity": 1440,            # ← in minutes: 1440 = daily, 720 = 12h
}

resp = requests.get(url, params=params).json()

# Debug: check what the API returns
print(resp.keys())
history = resp.get('history', resp)  # sometimes data is directly in root

In [ ]:
history[:5]

In [ ]:
df

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

# ─── STEP 1: Fetch event data ───
slug = "presidential-election-winner-2024"
gamma_url = f"https://gamma-api.polymarket.com/events?slug={slug}"
event = requests.get(gamma_url).json()[0]

# ─── STEP 2: Fetch token IDs for Trump and Harris ───
def get_yes_token(markets, name):
    market = next(m for m in markets if name in m['question'])
    token_ids = json.loads(market['clobTokenIds'])
    return token_ids[0]  # index 0 = Yes token

trump_token = get_yes_token(event['markets'], 'Trump')
harris_token = get_yes_token(event['markets'], 'Harris')

print(f"Trump token:  {trump_token}")
print(f"Harris token: {harris_token}")

# ─── STEP 3: Fetch price history ───
def get_price_history(token_id, label):
    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market": token_id,
        "interval": "max",
        "fidelity": 1440,  # daily
    }
    resp = requests.get(url, params=params).json()
    history = resp.get('history', [])
    
    df = pd.DataFrame(history)
    df['date'] = pd.to_datetime(df['t'], unit='s')
    df['price'] = df['p'].astype(float) * 100  # convert to percentage
    df['candidate'] = label
    
    # Deduplicate: max per day (filters Yes/No duplicates)
    df['date_only'] = df['date'].dt.date
    df = df.groupby('date_only')['price'].max().reset_index()
    df.columns = ['date', 'price']
    df['candidate'] = label
    return df

trump_df  = get_price_history(trump_token,  'Donald Trump')
harris_df = get_price_history(harris_token, 'Kamala Harris')

In [ ]:
# ─── Combine Trump and Harris into one DataFrame ───
# fidelity=1440 → one price point per day at 00:00 UTC
# 00:00 UTC of day D = closing price of day D-1 in US time (≈ 19:00-20:00 EST)
# → shift date back by 1 day so the label matches the actual US trading day

def get_price_history_full(token_id, label):
    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market": token_id,
        "interval": "max",
        "fidelity": 1440,  # one point per day at 00:00 UTC
    }
    resp = requests.get(url, params=params).json()
    history = resp.get('history', [])

    df = pd.DataFrame(history)
    df['datetime_utc'] = pd.to_datetime(df['t'], unit='s')
    df['probability']  = df['p'].astype(float) * 100

    # Correct date label: 00:00 UTC day D → closing price of day D-1 (US time)
    df['date'] = (df['datetime_utc'] - pd.Timedelta(days=1)).dt.normalize()

    return df[['date', 'probability']].rename(columns={'probability': label})

trump_df  = get_price_history_full(trump_token,  'Trump (%)')
harris_df = get_price_history_full(harris_token, 'Harris (%)')

# Merge on date (outer join to preserve all data)
combined_df = pd.merge(trump_df, harris_df, on='date', how='outer')
combined_df = combined_df.sort_values('date').reset_index(drop=True)

# Round to 2 decimal places
combined_df['Trump (%)']  = combined_df['Trump (%)'].round(2)
combined_df['Harris (%)'] = combined_df['Harris (%)'].round(2)

print(combined_df)
print(f"\nShape: {combined_df.shape}")

In [ ]:
# Filter from July 5 onwards (start of relevant campaign period)
final_dependent = combined_df[combined_df['date'] >= '2024-07-05'].reset_index(drop=True)
print(final_dependent)

In [ ]:
final_dependent.dtypes

In [ ]:
import os

output_path = "../../data/1_bronze/polymarket/polymarket_win_probabilities.csv"
final_dependent.to_csv(output_path, index=False)
print(f"Saved {len(final_dependent)} rows to {output_path}")

Get all candidates for Juli

In [ ]:
# ─── get probability history for all candidates ───
def get_price_history_full(token_id, label):
    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market": token_id,
        "interval": "max",
        "fidelity": 1440,
    }
    resp = requests.get(url, params=params).json()
    history = resp.get('history', [])
    if not history:
        return pd.DataFrame(columns=['date', label])
    
    df = pd.DataFrame(history)
    df['date'] = pd.to_datetime(df['t'], unit='s')
    df['date'] = df['date'].dt.normalize()  
    df = df.groupby('date')['p'].max().reset_index()
    df['p'] = df['p'].astype(float) * 100
    df = df.rename(columns={'p': label})
    return df

# Loop over all candidates
all_dfs = []
for market in event['markets']:
    try:
        name = market['question'].replace("Will ", "").replace(" win the 2024 US Presidential Election?", "")
        token_ids = json.loads(market['clobTokenIds'])
        yes_token = token_ids[0]
        df = get_price_history_full(yes_token, name)
        all_dfs.append(df)
        print(f"✓ {name}")
    except Exception as e:
        print(f"✗ {name}: {e}")

In [ ]:
# ─── All candidates in one dataframe ───
from functools import reduce

combined_all = reduce(lambda left, right: pd.merge(left, right, on='date', how='outer'), all_dfs)
combined_all = combined_all.sort_values('date').reset_index(drop=True)
combined_all = combined_all.round(2)

print(combined_all)

In [ ]:
# ─── STEP 1: Filter to July 2024 ───
july_df = combined_all[
    (combined_all['date'] >= '2024-07-01') & 
    (combined_all['date'] <= '2024-07-31')
].copy()

# ─── STEP 2: Find top candidates by mean win probability in July ───
# Get all candidate columns (everything except 'date')
candidate_cols = [col for col in july_df.columns if col != 'date']

# Calculate mean win probability per candidate in July
july_means = july_df[candidate_cols].mean()

# Pick top N candidates (e.g. top 6) by average probability
N = 6
top_candidates = july_means.nlargest(N).index.tolist()

print("Top candidates:", top_candidates)
print("\nJuly means:\n", july_means[top_candidates].round(2))

In [ ]:
output_path = "../../data/1_bronze/polymarket/polymarket_july.csv"
july_df.to_csv(output_path, index=False)
print(f"Saved {len(july_df)} rows to {output_path}")

Compare July dataframe with final dataset dataframe

In [ ]:
july_df.head(10)

In [ ]:
final_dependent.head()

---
## Visualisations

Source:  (merged here)

Plots: Trump win probability timeline, Trump vs Harris comparison, July top-candidates view.

In [ ]:
# Trump win probability – single-candidate view
plt.figure(figsize=(12, 5))
plt.plot(combined_df['date'], combined_df['Trump (%)'], color='red', linewidth=2)
plt.title("Trump win probability - Polymarket timeline")
plt.ylabel("Probability (%)")
plt.xlabel("Date")
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Trump vs Harris – win probability comparison
trump_df  = combined_df[['date', 'Trump (%)']].rename(columns={'Trump (%)': 'price'})
harris_df = combined_df[['date', 'Harris (%)']].rename(columns={'Harris (%)': 'price'})

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(trump_df['date'],  trump_df['price'],  color='red',  linewidth=2, label='Donald Trump')
ax.plot(harris_df['date'], harris_df['price'], color='blue', linewidth=2, label='Kamala Harris')

# Key events
key_events = {
    '2024-07-13': 'Assassination\nattempt Trump',
    '2024-07-21': 'Biden\ndrops out',
    '2024-08-19': 'Harris\nnomination',
    '2024-09-10': 'Debate\nTrump-Harris',
    '2024-11-05': 'Election\nDay',
}

for date_str, label in key_events.items():
    ax.axvline(pd.Timestamp(date_str), color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(pd.Timestamp(date_str), 72, label, rotation=90,
            fontsize=7.5, va='top', ha='right', color='gray')

ax.axhline(50, color='black', linestyle=':', alpha=0.4, linewidth=1)
ax.set_title("Trump vs Harris – Win probability on Polymarket (2024)", fontsize=14, fontweight='bold')
ax.set_ylabel("Probability (%)")
ax.set_xlabel("Date")
ax.set_ylim(10, 80)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Visualize top candidates in July ───
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))

colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']
for i, candidate in enumerate(top_candidates):
    ax.plot(july_df['date'], july_df[candidate], 
            label=f"{candidate} (avg: {july_means[candidate]:.1f}%)",
            linewidth=2, color=colors[i % len(colors)])

# Key events in july
ax.axvline(pd.Timestamp('2024-07-13'), color='gray', linestyle='--', alpha=0.6)
ax.text(pd.Timestamp('2024-07-13'), 5, 'Attack Trump', rotation=90, fontsize=8, color='gray')

ax.axvline(pd.Timestamp('2024-07-21'), color='gray', linestyle='--', alpha=0.6)
ax.text(pd.Timestamp('2024-07-21'), 5, 'Biden exit', rotation=90, fontsize=8, color='gray')

ax.set_title("Top kandidates July 2024 – Polymarket (99% coverage)", fontsize=13, fontweight='bold')
ax.set_ylabel("Winkans (%)")
ax.set_xlabel("Datum")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()